In [1]:
!pip install -r requirements.txt

ERROR: Could not find a version that satisfies the requirement os>=0.1.0 (from versions: none)
ERROR: No matching distribution found for os>=0.1.0


In [2]:
import mysql.connector
import numpy as np
import pandas as pd
import zarr
import os

In [3]:
HEIGHT = 256
WIDTH = 384

# パス設定
base_dir = "/Volumes/Transcend/zarrDB"
zarr_path = os.path.join(base_dir, "Haya2TIR.zarr")

# MySQL 接続
conn = mysql.connector.connect(
    user="root",
    password="",
    unix_socket="/tmp/mysql_database_dev.sock",
    database="heat_db",
)
cur = conn.cursor(dictionary=True)



In [4]:
cur.execute("SELECT * FROM tirimageinfo")
df = pd.DataFrame(cur.fetchall())

In [5]:
cur.close()
conn.close()


In [6]:
print(df.columns)

Index(['img_file', 'thumbnail', 'path', 'date_time', 'm', 'place',
       'target_name', 'phi', 'hood_t', 'target_t', 'len_t', 'plt_t', 'bol_t',
       'img_id', 'plt_t_set', 'bol_t_mon', 'pkg_t_mon', 'case_t_mon',
       'sh_t_mon', 'lens_t_mon'],
      dtype='object')


In [7]:
base_dir = "/Users/kengokakazu/研究用/HEAT/tileDB_migrate/zarrDB"
zarr_path = os.path.join(base_dir, "tirimageinfo.zarr")

In [8]:
root = zarr.open(zarr_path, mode="r")
pixel = root["pixel"]
pixel_modified = root["pixel_modified"]
mask = root["mask"]
filename_df = pd.read_csv(os.path.join(base_dir, "file_index_mapping.csv"))

In [ ]:
x,y = 17,17
m = filename_df.merge(df, left_on="filename", right_on="img_file", how="left")
m = m.sort_values("file_idx")
target_t = m["target_t"].to_numpy()
# ... 他列も配列で一括取得
vy = np.asarray(pixel_modified[:, y, x]) 
vx_df = m[["target_t", "bol_t_mon", "pkg_t_mon", "case_t_mon", "sh_t_mon", "lens_t_mon"]].rename(columns={"bol_t_mon":"bol_t","pkg_t_mon":"pkg_t","case_t_mon":"case_t","sh_t_mon":"sh_t","lens_t_mon":"lens_t"})

In [10]:
K_OFFSET = 273.15
header = vx_df.columns.tolist()
for c in header:
    vx_df[c] = vx_df[c] + K_OFFSET